In [1]:
# Enables IPython autoreload (two magic commands, the second takes a numeric argument)
%load_ext autoreload
%autoreload 2

import logging
import os
import time

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s  - %(name)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 20)
pd.set_option("display.float_format", "{:.6f}".format)


logger.info("Notebook initialized")

2026-05-01 23:38:47,476  - __main__ - INFO - Notebook initialized


---

## Final pipeline using `src/data.py`

The above cells walked through the exploration step-by-step. The same pipeline is now packaged into two functions in `src/data.py`. The cells below verify that the imported functions produce the same output as the inline exploration.

In [2]:
from src.data import compute_returns, download_prices

prices = download_prices()
returns = compute_returns(prices)

print("Daily prices shape:", prices.shape)
print("Monthly returns shape:", returns.shape)
print("\nFirst 3 returns:")
print(returns.head(3))
print("\nLast 3 returns:")
print(returns.tail(3))

2026-05-01 23:38:47,537  - src.data - INFO - Loading prices from cache: /Users/tangentjet/Projects/mean-variance-portfolio/data/raw/prices_daily.parquet
2026-05-01 23:38:47,617  - src.data - INFO - Computed monthly returns: (120, 8)


Daily prices shape: (2538, 8)
Monthly returns shape: (120, 8)

First 3 returns:
                 SPY      GOVT     EEMV       CME       BR      CBOE  \
date                                                                   
2015-01-31 -0.029629  0.029423 0.011831 -0.037789 0.039194  0.016556   
2015-02-28  0.056205 -0.017439 0.025305  0.124619 0.109189 -0.065708   
2015-03-31 -0.015745  0.006021 0.004426 -0.007544 0.038856 -0.043728   

                 ICE       ACN  
date                            
2015-01-31 -0.061836 -0.059120  
2015-02-28  0.144024  0.071403  
2015-03-31 -0.006068  0.040653  

Last 3 returns:
                 SPY      GOVT      EEMV      CME        BR      CBOE  \
date                                                                    
2024-10-31 -0.008924 -0.024286 -0.037161 0.021346 -0.019393  0.042466   
2024-11-30  0.059634  0.008489 -0.008945 0.056088  0.119321  0.013626   
2024-12-31 -0.024100  0.006945 -0.007585 0.004852 -0.038463 -0.094742   

           

In [3]:
from src.data import compute_returns, download_prices
from src.stats import arithmetic_mean, geometric_mean

prices = download_prices()
returns = compute_returns(prices)

print("Arithmetic mean (monthly):")
print(arithmetic_mean(returns))
print("\nGeometric mean (monthly):")
print(geometric_mean(returns))

2026-05-01 23:38:47,634  - src.data - INFO - Loading prices from cache: /Users/tangentjet/Projects/mean-variance-portfolio/data/raw/prices_daily.parquet
2026-05-01 23:38:47,637  - src.data - INFO - Computed monthly returns: (120, 8)


Arithmetic mean (monthly):
SPY    0.011216
GOVT   0.000917
EEMV   0.002965
CME    0.012852
BR     0.016831
CBOE   0.012521
ICE    0.013060
ACN    0.015002
dtype: float64

Geometric mean (monthly):
SPY    0.010243
GOVT   0.000815
EEMV   0.002326
CME    0.011446
BR     0.014829
CBOE   0.010574
ICE    0.011327
ACN    0.012883
dtype: float64


In [4]:
from src.data import compute_returns, download_prices
from src.stats import arithmetic_mean, covariance_matrix, geometric_mean, volatility

prices = download_prices()
returns = compute_returns(prices)

print("Arithmetic mean (monthly):")
print(arithmetic_mean(returns))
print("\nVolatility (monthly):")
print(volatility(returns))
print("\nCovariance matrix (monthly):")
print(covariance_matrix(returns))
print("\nDiagonal of covariance vs volatility squared:")
print((volatility(returns) ** 2).round(8))
print(np.diag(covariance_matrix(returns)).round(8))

2026-05-01 23:38:47,651  - src.data - INFO - Loading prices from cache: /Users/tangentjet/Projects/mean-variance-portfolio/data/raw/prices_daily.parquet
2026-05-01 23:38:47,654  - src.data - INFO - Computed monthly returns: (120, 8)


Arithmetic mean (monthly):
SPY    0.011216
GOVT   0.000917
EEMV   0.002965
CME    0.012852
BR     0.016831
CBOE   0.012521
ICE    0.013060
ACN    0.015002
dtype: float64

Volatility (monthly):
SPY    0.044245
GOVT   0.014359
EEMV   0.035842
CME    0.053643
BR     0.064040
CBOE   0.062143
ICE    0.059594
ACN    0.065292
dtype: float64

Covariance matrix (monthly):
          SPY      GOVT     EEMV       CME       BR     CBOE      ICE      ACN
SPY  0.001958  0.000083 0.001100  0.000858 0.001865 0.000937 0.001755 0.002302
GOVT 0.000083  0.000206 0.000106 -0.000023 0.000229 0.000036 0.000168 0.000183
EEMV 0.001100  0.000106 0.001285  0.000315 0.000910 0.000341 0.000783 0.001126
CME  0.000858 -0.000023 0.000315  0.002878 0.001094 0.001688 0.001821 0.001110
BR   0.001865  0.000229 0.000910  0.001094 0.004101 0.001098 0.001957 0.002774
CBOE 0.000937  0.000036 0.000341  0.001688 0.001098 0.003862 0.001688 0.001430
ICE  0.001755  0.000168 0.000783  0.001821 0.001957 0.001688 0.003551 0.002368
AC

In [5]:
from src.data import compute_returns, download_prices
from src.optimizer import solve_mvo_scipy
from src.stats import arithmetic_mean, covariance_matrix

prices = download_prices()
returns = compute_returns(prices)
mu = arithmetic_mean(returns)
sigma = covariance_matrix(returns)

# Try a target return that should be feasible: middle of the per-asset range
target = 0.01  # 1% monthly target

weights = solve_mvo_scipy(mu, sigma, target_return=target)
print("Weights:")
print(weights)
print(f"\nSum of weights: {weights.sum():.6f}")
print(f"Portfolio expected return: {(mu * weights).sum():.6f}")
print(f"Portfolio variance: {weights @ sigma @ weights:.6f}")
print(f"Portfolio volatility: {(weights @ sigma @ weights)**0.5:.6f}")

2026-05-01 23:38:48,703  - src.data - INFO - Loading prices from cache: /Users/tangentjet/Projects/mean-variance-portfolio/data/raw/prices_daily.parquet
2026-05-01 23:38:48,708  - src.data - INFO - Computed monthly returns: (120, 8)
2026-05-01 23:38:48,714  - src.optimizer - INFO - scipy MVO converged: variance=0.000957, return=0.010000


Weights:
SPY    0.201598
GOVT   0.272333
EEMV   0.000000
CME    0.235232
BR     0.191283
CBOE   0.099554
ICE    0.000000
ACN    0.000000
Name: weights, dtype: float64

Sum of weights: 1.000000
Portfolio expected return: 0.010000
Portfolio variance: 0.000957
Portfolio volatility: 0.030928


In [6]:
from src.data import compute_returns, download_prices
from src.optimizer import solve_mvo_cvxpy, solve_mvo_scipy
from src.stats import arithmetic_mean, covariance_matrix

prices = download_prices()
returns = compute_returns(prices)
mu = arithmetic_mean(returns)
sigma = covariance_matrix(returns)

target = 0.01
weights_scipy = solve_mvo_scipy(mu, sigma, target_return=target)
weights_cvxpy = solve_mvo_cvxpy(mu, sigma, target_return=target)

print("scipy weights:")
print(weights_scipy.round(4))
print("\ncvxpy weights:")
print(weights_cvxpy.round(4))
print("\nMax absolute difference:")
print((weights_scipy - weights_cvxpy).abs().max())

2026-05-01 23:38:48,740  - src.data - INFO - Loading prices from cache: /Users/tangentjet/Projects/mean-variance-portfolio/data/raw/prices_daily.parquet
2026-05-01 23:38:48,744  - src.data - INFO - Computed monthly returns: (120, 8)
2026-05-01 23:38:48,749  - src.optimizer - INFO - scipy MVO converged: variance=0.000957, return=0.010000
2026-05-01 23:38:48,757  - src.optimizer - INFO - cvxpy MVO converged: variance=0.000957, return=0.010000, status=optimal


scipy weights:
SPY    0.201600
GOVT   0.272300
EEMV   0.000000
CME    0.235200
BR     0.191300
CBOE   0.099600
ICE    0.000000
ACN    0.000000
Name: weights, dtype: float64

cvxpy weights:
SPY    0.201600
GOVT   0.272300
EEMV   0.000000
CME    0.235300
BR     0.191200
CBOE   0.099500
ICE    0.000000
ACN    0.000000
Name: weights, dtype: float64

Max absolute difference:
4.847979005434544e-05


In [7]:
from src.data import compute_returns, download_prices
from src.optimizer import solve_mvo_cvxpy, solve_mvo_scipy
from src.stats import arithmetic_mean, covariance_matrix

prices = download_prices()
returns = compute_returns(prices)
mu = arithmetic_mean(returns)
sigma = covariance_matrix(returns)

target = 0.01
ws = solve_mvo_scipy(mu, sigma, target_return=target)
wc = solve_mvo_cvxpy(mu, sigma, target_return=target)

# Compute variance of each portfolio under the same Σ
var_s = ws @ sigma @ ws
var_c = wc @ sigma @ wc

print(f"scipy variance:  {var_s:.10f}")
print(f"cvxpy variance:  {var_c:.10f}")
print(f"scipy return:    {(mu * ws).sum():.10f}")
print(f"cvxpy return:    {(mu * wc).sum():.10f}")
print(f"scipy sum:       {ws.sum():.10f}")
print(f"cvxpy sum:       {wc.sum():.10f}")

2026-05-01 23:38:48,791  - src.data - INFO - Loading prices from cache: /Users/tangentjet/Projects/mean-variance-portfolio/data/raw/prices_daily.parquet
2026-05-01 23:38:48,807  - src.data - INFO - Computed monthly returns: (120, 8)
2026-05-01 23:38:48,820  - src.optimizer - INFO - scipy MVO converged: variance=0.000957, return=0.010000
2026-05-01 23:38:48,827  - src.optimizer - INFO - cvxpy MVO converged: variance=0.000957, return=0.010000, status=optimal


scipy variance:  0.0009565115
cvxpy variance:  0.0009565115
scipy return:    0.0100000000
cvxpy return:    0.0100000000
scipy sum:       1.0000000000
cvxpy sum:       1.0000000000
